# 5 mmol Static XAS Peak Significance

Pooled XAS, smooth-continuum residuals, and shot-level bootstrap uncertainty for checking whether small spectral bumps are larger than random shot noise.

In [ ]:
# Cell 1: config only
from __future__ import annotations

import os
import sys
import tempfile
from pathlib import Path

mpl_cache = Path(tempfile.gettempdir()) / "glycine26-matplotlib-cache"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160})


def find_repo_root(start: Path) -> Path:
    for parent in (start.resolve(), *start.resolve().parents):
        marker = parent / "analysis" / "scripts" / "static_xas_peak_significance.py"
        if marker.exists():
            return parent
    raise RuntimeError("Could not find repo root from notebook path.")


REPO_ROOT = find_repo_root(Path.cwd())
SCRIPTS_DIR = REPO_ROOT / "analysis" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import config as path_config
import static_xas_peak_significance as peak


INPUT_H5 = Path(path_config.COMBINED_DIR).parent / "xas_static" / "run58794_static_xas.h5"

# Shot filtering
VLS_INTENSITY_THRESHOLD = 1000.0
GMD_MIN_THRESHOLD = None
GMD_MAX_THRESHOLD = 10.0

# Central photon-energy proxy from each VLS spectrum
CENTER_METHOD = "com"  # "com", "peak", "smooth_peak", "gaussian_logfit"
CENTER_HALF_WINDOW = 5
CENTER_CHUNK_SIZE = 50_000
PROXY_PIXEL_BIN_WIDTH = 1

# Thesis-style signal and optional per-nominal-energy intercept correction
CORRECTION_MODE = "per_energy_intercept"  # "none" or "per_energy_intercept"
XAS_Y_MODE = "od"  # "od" = log10[sum(GMD)/sum(VLS)], or "ratio"

# Smooth continuum and bootstrap uncertainty
SMOOTH_METHOD = "savgol"
SMOOTH_WINDOW_EV = 1.5
SMOOTH_POLYORDER = 2
N_BOOTSTRAP = 500
BOOTSTRAP_SEED = 123
MIN_SHOTS_PER_POINT = 100
BOOTSTRAP_BATCH = 50

# Plot controls
XAS_XLIM = None
OD_YLIM = None
RESIDUAL_YLIM = None
FEATURE_WINDOWS_EV = []  # e.g. [(273.8, 274.2), (275.4, 275.8)]
Z_MARKERS = (2.0, 3.0)

# Advanced: pixel-to-energy map. Defaults match the current simple notebook.
PIXEL_MAP_CONFIG = {
    "source_to_grating_m": 0.85,
    "grating_to_screen_m": 0.5429,
    "incident_angle_deg": 86.78,
    "groove_density_lines_per_mm": 1200.0,
    "diffraction_order": 1,
    "magnification": 3.0,
    "pixel_pitch_m": 50e-6,
    "axis_sign": 1,
}
PIXEL_MAP_CALIBRATION = {"pixel": 524.0, "energy_eV": 284.2}
INVERSE_MAP_RANGE_EV = (260.0, 290.0)
INVERSE_MAP_SAMPLES = 10_000

CHUNK_SIZE = 20_000

CONFIG = peak.PeakSignificanceConfig(
    input_h5=INPUT_H5,
    vls_intensity_threshold=VLS_INTENSITY_THRESHOLD,
    gmd_min_threshold=GMD_MIN_THRESHOLD,
    gmd_max_threshold=GMD_MAX_THRESHOLD,
    center_method=CENTER_METHOD,
    center_half_window=CENTER_HALF_WINDOW,
    center_chunk_size=CENTER_CHUNK_SIZE,
    proxy_pixel_bin_width=PROXY_PIXEL_BIN_WIDTH,
    correction_mode=CORRECTION_MODE,
    xas_y_mode=XAS_Y_MODE,
    smooth_method=SMOOTH_METHOD,
    smooth_window_ev=SMOOTH_WINDOW_EV,
    smooth_polyorder=SMOOTH_POLYORDER,
    n_bootstrap=N_BOOTSTRAP,
    bootstrap_seed=BOOTSTRAP_SEED,
    min_shots_per_point=MIN_SHOTS_PER_POINT,
    bootstrap_batch=BOOTSTRAP_BATCH,
    chunk_size=CHUNK_SIZE,
    feature_windows_ev=FEATURE_WINDOWS_EV,
    z_markers=Z_MARKERS,
    pixel_map_config=PIXEL_MAP_CONFIG,
    pixel_map_calibration=PIXEL_MAP_CALIBRATION,
    inverse_map_range_ev=INVERSE_MAP_RANGE_EV,
    inverse_map_samples=INVERSE_MAP_SAMPLES,
)

In [ ]:
# Cell 2: compute significance result
result = peak.compute_peak_significance(CONFIG)
nominal_counts = result.nominal.counts[np.isfinite(result.nominal.xas)]
converted_counts = result.converted.counts[np.isfinite(result.converted.xas)]

print(f"GMD source: {result.gmd_source}")
print(f"accepted shots: {int(np.sum(result.filter_counts['accepted'])):,}")
print(f"nominal finite residual bins: {int(np.isfinite(result.nominal.residual).sum())}")
print(f"converted finite residual bins: {int(np.isfinite(result.converted.residual).sum())}")
print(f"nominal bootstrap shots/bin min/median/max: {np.min(nominal_counts):,} / {np.median(nominal_counts):,.0f} / {np.max(nominal_counts):,}")
print(f"converted bootstrap shots/bin min/median/max: {np.min(converted_counts):,} / {np.median(converted_counts):,.0f} / {np.max(converted_counts):,}")

In [ ]:
# Cell 3: final plot
peak.plot_peak_significance_summary(
    result,
    xlim=XAS_XLIM,
    od_ylim=OD_YLIM,
    residual_ylim=RESIDUAL_YLIM,
)
plt.show()